# Cryptography (CC4017) -- Week 8

## Q1: Implementing Authenticated Encryption

Implement a prototype that exemplifies the behavior of an authenticated encryption scheme. Your goal
is to ensure secure communications between Alice and Bob. The system must ensure the following:
- Message confidentiality. Use AES-128-CTR to do this
- Message authenticity. Use HMAC-SHA256 to do this
- Protection against replay attacks. Include a sequence number to the messages sent

As such, this task requires you to implement alice.py and bob.py, communicating using authenticated
encryption, supported by gen.py, which generates the pre-shared keys. Use Encrypt-then-MAC,
meaning that you should calculate HMAC from the result of AES-CTR, and send both of these values
over the network. The tasks are as follows:

1. Upon execution, gen.py must produce a file pw with two symmetric keys. One to be used for
encryption, the other to be used for message authentication
2. Upon execution, alice.py and bob.py must begin by reading the file pw to get their keys.
3. Then, alice.py and bob.py must exchange the following messages:
- From Alice: “Hello Bob”
- From Bob: “Hello Alice”
- From Alice: “I would like to have dinner”
- From Bob: "Me too. Same time, same place?
- From Alice: “Sure!”

Students are encouraged to tackle this challenge one step at a time. The suggested stages as as
follows:

Part 1: Implement gen.py and test if alice.py and bob.py are reading the keys correctly.

Part 2: Implement the communication layer between Alice and Bob using sockets (hint: check out
this reference), or pwntools (reference). Test if you can send bytes and if they are arriving without
errors.

Part 3: Adapt the messages sent to now be the result of AES-128-CTR. See if the decryption is
successful.

Part 4: Include the result of HMAC-SHA256 in the sent message. See if the authentication is
successful.

Part 5: Include a sequence number in both alice.py and bob.py, and append it to the sent message.
Check if everything is working.

Final: Adapt your prototype to have Alice and Bob send the specified messages. Check if everything
is validated, and correctly decrypted.

The encryption is done using AES_CTR, as such it needs a nonce value. As alice starts the connection with Bob, who, due to standad sockets python workage behaves as the server, they both initiate a message counter as 1 and 0 respectively, since Alice is the one who sends the first message. This counter is passed alongside the ciphered message and the nonce. As such an header with the size of the nonce and containing the counter is included forming a packet of header+ciphertext+nonce. In order to prevent tampering with the packet, the whole packet is signed using hmac, the signature is prepended to the packet and sent alongsides it. When receiving it the user reads the signature and compares it to the one expected by signing the packet received themselves, they then extract the 8 byte header, obtaining the counther the nonce and the ciphertext, that is in its own turn deciphered. The counter behaves as the sequence number in the messages, increasing and guaranteeing defense alongsides replay attacks. If at any time a tampered message is detected, like a replay attack or a unauthenticated message being sent, the communication is terminated by the receiver of the message.

To correctly test the program, first execute bob.py and then alice.py



## Q2: Signing with RSA


### P1: 
Let d denote the private key and e denote the public key for RSA, m denote the message we want to
sign and σ denote the produced signature. A naive way to use RSA for digital signatures is to simply
encrypt the message using the private key. Consider the following signature scheme:

- Sign: σ ← M^d mod N
- Verify: Compute M' ← σ^e mod N . Accept if M = M'

Show how this signature can never be shown to be unforgeable, by constructing a valid signature for a message without knowledge of the private key d.


One simple forgeable signature that can be done is sending the tuple (1,1). Without knowing d, we are certain that 1^d mod N will always be 1, and 1^e will also be 1, as such we can construct a valid pair of message signature. Another one of these possible bad numbers is 0, the tuple (0,0) will always form a valid tuple to be sent and verified, since 0^d is 0 and 0^e is also 0.

What an attacker might due instead is pick a random number, that will take the place of σ. In this case, since e is publically availlable they calculate X = σ^e mod N, and send to the target σ and X. Since the way to obtain X and M' are the same, and both will use sigma, an attacker can send a message that is signed, without ever needing to use d.


### P2:
Full Domain Hash (FDH) are constructions that also rely on RSA to produce digital signatures, but make use of a cryptographic hash function (H) to avoid these issues. FDH behaves as follows:
- Sign: Compute h ← H(M), and σ ← h^d mod N
- Verify: Compute h' ← σ^e mod N . Accept if H(M) = h'

What properties of the hash functions are we using to ensure that the previous attack no longer works?

Pre image resistence makes it so the last attack mentioned no longer works. We might suggest a random σ value to to encrypt and obtain h'. However, since we are unnable to predict what the expected value of H(m) is, its infeasible to admit that an attacker can successfully both pick a random σ corresponding to its hash value as well as the the message that computes that σ value. Collision resistence makes it so its harded to produce values that an attacker knows to have the same signature values, stopping them from generating valid signature message pairs easily